# Zipline International

# Analysis of Pricing and Session Data

## Setup

In [27]:
#load libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print("Setup complete!")

Setup complete!


## Context

**Data definitions**

**Pricing dataset**  
zipline_pricing_schedule.csv  
For 30 days, we ran a very random delivery fee test. This file indicates the delivery fee we showed to customers at any given time for all of the 15 hubs

*Columns:*  
nest_id = hub id for drone home location  
date_hour = timestamp in ?15 min increments (need to confirm)  
delivery_fee = delivery fee we showed to customers at any given time for all of the 15 hub  


**Session dataset**  
zipline_session_log.csv  
Session data for the same 30-day period  

*Columns:*  
timestamp = timestamp of order - beginning or end?  
user_id = unique user  
nest_id = hub id for drone home location  
order_type = one of 3 possible values: 'Grocery', 'Meal', 'Pharma'  
distance_miles = distance from hub to delivery location (one-way?)  
delivery_fee = delivery fee charged  
subtotal = amount paid by user (excluding delivery fee?)  
session_outcome = one of thee possible values: 'abandoned', 'blocked', 'ordered'

## Load data

In [5]:
#Load datasets
pricing_df = pd.read_csv('../data/raw/zipline_pricing_schedule.csv')
session_df = pd.read_csv('../data/raw/zipline_session_log_(5).csv')

print("✓ Data loaded successfully!")

✓ Data loaded successfully!


### Pricing Data

In [37]:
# Display first and last rows of pricing data
display(pricing_df.head())
display(pricing_df.tail())
display(pricing_df.dtypes)

,nest_id,date_hour,delivery_fee
0,0,2024-01-01 08:00:00,2.99
1,1,2024-01-01 08:00:00,4.99
2,2,2024-01-01 08:00:00,0.99
3,3,2024-01-01 08:00:00,3.99
4,4,2024-01-01 08:00:00,3.99


,nest_id,date_hour,delivery_fee
21490,10,2024-03-01 00:00:00,3.99
21491,11,2024-03-01 00:00:00,4.99
21492,12,2024-03-01 00:00:00,3.99
21493,13,2024-03-01 00:00:00,2.99
21494,14,2024-03-01 00:00:00,3.99


nest_id           int64
date_hour        object
delivery_fee    float64
dtype: object

In [36]:
#pricing data stats
display(pricing_df.describe())
display(pricing_df.info())

,nest_id,delivery_fee
count,21495.000000,21495.000000
mean,7.000000,3.658062
std,4.320594,1.971848
min,0.000000,0.990000
25%,3.000000,1.990000
50%,7.000000,3.990000
75%,11.000000,4.990000
max,14.000000,6.990000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21495 entries, 0 to 21494
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   nest_id       21495 non-null  int64  
 1   date_hour     21495 non-null  object 
 2   delivery_fee  21495 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 503.9+ KB


None

### Session Data

In [35]:
#session data raw
display(session_df.head())
display(session_df.tail())
display(session_df.dtypes)

,timestamp,user_id,nest_id,order_type,distance_miles,delivery_fee,subtotal,session_outcome
0,1/1/24 8:01,1746,9,Meal,7.3,2.99,44.36,ordered
1,1/1/24 8:01,3208,8,Grocery,4.4,NaN,NaN,abandoned
2,1/1/24 8:04,3054,4,Meal,8.7,NaN,NaN,abandoned
3,1/1/24 8:05,806,1,Meal,11.8,NaN,NaN,abandoned
4,1/1/24 8:05,1731,11,Meal,6.1,0.99,29.12,ordered


,timestamp,user_id,nest_id,order_type,distance_miles,delivery_fee,subtotal,session_outcome
59875,1/13/24 11:01,3634,4,Meal,9.2,6.99,29.68,ordered
59876,1/13/24 11:01,2164,5,Meal,11.8,0.99,19.18,ordered
59877,1/13/24 11:01,540,4,Meal,4.1,6.99,20.53,ordered
59878,1/13/24 11:01,4087,14,Meal,2.1,NaN,NaN,abandoned
59879,1/13/24 11:01,3863,12,Meal,6.9,NaN,NaN,abandoned


timestamp           object
user_id              int64
nest_id              int64
order_type          object
distance_miles     float64
delivery_fee       float64
subtotal           float64
session_outcome     object
dtype: object

In [28]:
#session data stats
display(pricing_df.describe())
display(pricing_df.info())

,nest_id,delivery_fee
count,21495.000000,21495.000000
mean,7.000000,3.658062
std,4.320594,1.971848
min,0.000000,0.990000
25%,3.000000,1.990000
50%,7.000000,3.990000
75%,11.000000,4.990000
max,14.000000,6.990000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21495 entries, 0 to 21494
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   nest_id       21495 non-null  int64  
 1   date_hour     21495 non-null  object 
 2   delivery_fee  21495 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 503.9+ KB


None

In [38]:
# Identify potential join keys by looking for common column names
pricing_cols = set(pricing_df.columns)
session_cols = set(session_df.columns)
common_cols = pricing_cols.intersection(session_cols)

print("POTENTIAL JOIN KEYS (Common Columns)")
print("=" * 80)
if common_cols:
    print(f"Common columns found: {sorted(common_cols)}")
else:
    print("No common column names found.")
    print("\nPricing columns:", sorted(pricing_cols))
    print("Session columns:", sorted(session_cols))

POTENTIAL JOIN KEYS (Common Columns)
Common columns found: ['delivery_fee', 'nest_id']


## Data Structure

In [41]:
# Examine unique values in key columns to understand structure
print("PRICING DATA STRUCTURE")

for col in pricing_df.columns:
    n_unique = pricing_df[col].nunique()
    print(f"\nPricing - '{col}':")
    print(f"  Unique values: {n_unique}")
    if n_unique <= 20:  # Show unique values if 20 or fewer
        print(f"  Values: {sorted(pricing_df[col].unique())}")
    else:
        print(f"  Sample values: {pricing_df[col].head(3).tolist()}")

PRICING DATA STRUCTURE

Pricing - 'nest_id':
  Unique values: 15
  Values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

Pricing - 'date_hour':
  Unique values: 1433
  Sample values: ['2024-01-01 08:00:00', '2024-01-01 08:00:00', '2024-01-01 08:00:00']

Pricing - 'delivery_fee':
  Unique values: 6
  Values: [0.99, 1.99, 2.99, 3.99, 4.99, 6.99]


In [42]:
# Examine unique values in key columns to understand structure
print("SESSION DATA STRUCTURE")

for col in session_df.columns:
    n_unique = session_df[col].nunique()
    print(f"\nSession - '{col}':")
    print(f"  Unique values: {n_unique}")
    if n_unique <= 20:
        print(f"  Values: {sorted(session_df[col].unique())}")
    else:
        print(f"  Sample values: {session_df[col].head(3).tolist()}")

SESSION DATA STRUCTURE

Session - 'timestamp':
  Unique values: 15759
  Sample values: ['1/1/24 8:01', '1/1/24 8:01', '1/1/24 8:04']

Session - 'user_id':
  Unique values: 5000
  Sample values: [1746, 3208, 3054]

Session - 'nest_id':
  Unique values: 15
  Values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

Session - 'order_type':
  Unique values: 3
  Values: ['Grocery', 'Meal', 'Pharma']

Session - 'distance_miles':
  Unique values: 101
  Sample values: [7.3, 4.4, 8.7]

Session - 'delivery_fee':
  Unique values: 6
  Values: [2.99, nan, 0.99, 1.99, 3.99, 4.99, 6.99]

Session - 'subtotal':
  Unique values: 4588
  Sample values: [44.36, nan, nan]

Session - 'session_outcome':
  Unique values: 3
  Values: ['abandoned', 'blocked', 'ordered']


## Exploratory Data Analysis

## Questions to answer

**QUESTIONS**

Do delivery fee values in *pricing* match delivery fee values paid in *session*, for matching times?

